# Data Science Programming - Sesion 5: Matplotlib y Seaborn

**Especializacion en Ciencia de Datos - Universidad Santo Tomas - Tunja**
**Prof. Hugo Hernandez | Martes 22 de septiembre, 2026**

---

## Que haremos hoy

1. **Por que graficar**: el cuarteto de Anscombe
2. **La anatomia de una figura**: `fig` y `ax`
3. **Los cinco graficos basicos** y que pregunta responde cada uno
4. **Personalizacion**: titulo, etiquetas, leyenda, colores, `bar_label`
5. **Del groupby al grafico**: los datos del Titanic
6. **Seaborn**: lo mismo con menos codigo
7. **El grafico que cuenta toda la historia**

> Ejecuta las celdas en orden con `Shift + Enter`.
> Experimenta: cambia colores y valores y vuelve a ejecutar. Nada se rompe para siempre.

---
## Antes de empezar

- **Matplotlib** es la libreria base con la que Python dibuja. Da control total.
- **Seaborn** esta construida encima de matplotlib: hace en una linea lo que en matplotlib
  toma veinte. Se pueden mezclar en la misma figura.
- En **VS Code / Jupyter**, ejecuta la celda de abajo una vez para instalar las librerias.

In [ ]:
%pip install matplotlib seaborn pandas numpy --quiet
print("Librerias listas")

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

print('matplotlib', matplotlib.__version__)
print('seaborn   ', sns.__version__)
print('pandas    ', pd.__version__)

---
## Parte 1 - Por que graficar: el cuarteto de Anscombe

Cuatro conjuntos de once puntos con las mismas estadisticas. Primero los numeros,
despues los dibujos.

In [ ]:
x_comun = [10.0, 8.0, 13.0, 9.0, 11.0, 14.0, 6.0, 4.0, 12.0, 7.0, 5.0]

anscombe = pd.DataFrame({
    'grupo': ['I'] * 11 + ['II'] * 11 + ['III'] * 11 + ['IV'] * 11,
    'x': x_comun + x_comun + x_comun + [8.0] * 7 + [19.0] + [8.0] * 3,
    'y': [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68] +
         [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74] +
         [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73] +
         [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]
})

print('Filas totales:', len(anscombe))
print(anscombe.head(3))

### 1.1 Primero los numeros

In [ ]:
resumen = anscombe.groupby('grupo').agg(
    media_x=('x', 'mean'),
    media_y=('y', 'mean'),
    desv_x=('x', 'std'),
    desv_y=('y', 'std')
)
print(resumen.round(2))
print()

for grupo, datos in anscombe.groupby('grupo'):
    correlacion = datos['x'].corr(datos['y'])
    print(f'Grupo {grupo:4} correlacion x-y = {correlacion:.3f}')

### 1.2 Ahora los dibujos

In [ ]:
sns.lmplot(data=anscombe, x='x', y='y', col='grupo', col_wrap=2, height=3)
plt.show()

### 1.3 La leccion

| Grupo | Que es en realidad |
|---|---|
| I | Una relacion lineal con ruido |
| II | Una **curva**, no una recta |
| III | Una recta perfecta con **un punto atipico** que la tuerce |
| IV | Todos los puntos en la misma x, salvo **uno** que inventa la correlacion |

> **Las estadisticas resumen; los graficos revelan.** Antes de reportar una estadistica,
> mira la forma.

---
## Parte 2 - La anatomia de una figura: `fig` y `ax`

| Objeto | Que es | Analogia |
|---|---|---|
| `fig` | La figura completa, el area que se guarda como imagen | La hoja de papel |
| `ax` | Los ejes: el marco donde se dibujan los datos | El marco sobre la hoja |

Una `fig` puede contener varios `ax`. Lo de la hoja se le pide a `fig`; lo del contenido,
a `ax`.

In [ ]:
meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun']
ventas = [120, 135, 100, 160, 180, 150]

fig, ax = plt.subplots(figsize=(8, 4))

print('fig es:', type(fig))
print('ax  es:', type(ax))

ax.plot(meses, ventas, marker='o', color='#1A56DB')

ax.set_title('Ventas mensuales de la cafeteria')
ax.set_xlabel('Mes')
ax.set_ylabel('Ventas (millones)')

plt.show()

### 2.1 Por que `fig, ax = plt.subplots()` y no `plt.plot()` a secas

| Estilo | Que hace por dentro |
|---|---|
| `plt.plot(...)` | Dibuja sobre "la figura actual", una variable escondida |
| `ax.plot(...)` | Dibuja sobre el objeto que tu tienes en la mano |

**Norma del curso: siempre `fig, ax = plt.subplots()`.**

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(meses, ventas)
plt.title('Estilo plt: funciona, pero no se sobre que estoy dibujando')
plt.show()

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(meses, ventas)
ax.set_title('Estilo ax: se exactamente sobre que dibujo')
plt.show()

### 2.2 Varios `ax` en una misma `fig`

`plt.subplots(1, 2)` = una fila, dos columnas. Devuelve una coleccion de ejes:
`ejes[0]`, `ejes[1]`.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))

print('Ahora el segundo valor es:', type(ejes))
print('Cuantos ejes hay:', len(ejes))

ejes[0].plot(meses, ventas, marker='o', color='#1A56DB')
ejes[0].set_title('Como linea')

ejes[1].bar(meses, ventas, color='#0891B2')
ejes[1].set_title('Como barras')

fig.suptitle('Los mismos datos, dos lecturas distintas')

plt.tight_layout()
plt.show()

### 2.3 La regla que matplotlib nunca perdona

In [ ]:
ventas_incompletas = [120, 135, 100, 160, 180]

print('Cuantos meses:', len(meses))
print('Cuantas ventas:', len(ventas_incompletas))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(meses, ventas_incompletas, marker='o')
ax.set_title('Ventas mensuales de la cafeteria')
plt.show()

---
## Parte 3 - Cinco graficos y la pregunta que responde cada uno

| Quieres saber... | Grafico | Metodo de `ax` | Funcion de seaborn |
|---|---|---|---|
| Cual categoria es mas grande | **Barras** | `ax.bar()` | `sns.barplot()` |
| Como evoluciona algo en el tiempo | **Linea** | `ax.plot()` | `sns.lineplot()` |
| Si dos variables numericas se mueven juntas | **Dispersion** | `ax.scatter()` | `sns.scatterplot()` |
| Como se reparte una sola variable numerica | **Histograma** | `ax.hist()` | `sns.histplot()` |
| Como se comparan grupos y donde hay atipicos | **Caja** | `ax.boxplot()` | `sns.boxplot()` |

In [ ]:
categorias = ['Bebidas', 'Panaderia', 'Postres', 'Sanduches']
unidades = [420, 310, 180, 260]

generador = np.random.default_rng(7)
gasto = generador.normal(18000, 5000, 200).round(0)
minutos = (gasto / 1000 + generador.normal(0, 3, 200)).round(1)

print('Categorias:', categorias)
print('Primeros cinco gastos:', gasto[:5])

### 3.1 Barras: comparar categorias

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(categorias, unidades, color='#1A56DB')
ax.set_title('Unidades vendidas por categoria')
ax.set_ylabel('Unidades')
plt.show()

### 3.2 Linea: evolucion en el tiempo

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(meses, ventas, marker='o', linewidth=2, markersize=7, color='#0891B2')
ax.set_title('Evolucion de las ventas, enero a junio')
ax.set_xlabel('Mes')
ax.set_ylabel('Ventas (millones)')
ax.grid(alpha=0.3)
plt.show()

### 3.3 Dispersion: relacion entre dos variables numericas

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(gasto, minutos, alpha=0.6, color='#7C3AED', edgecolors='white')
ax.set_title('Gasto contra tiempo de permanencia')
ax.set_xlabel('Gasto del cliente (pesos)')
ax.set_ylabel('Minutos en el local')
ax.grid(alpha=0.3)
plt.show()

### 3.4 Histograma: la distribucion de una sola variable

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(gasto, bins=15, color='#0891B2', edgecolor='white')
ax.set_title('Como se reparte el gasto de los clientes')
ax.set_xlabel('Gasto (pesos)')
ax.set_ylabel('Cuantos clientes')
plt.show()

### 3.5 Caja (boxplot): comparar grupos y cazar atipicos

La caja va del primer al tercer cuartil, la raya es la mediana, los puntos sueltos son
los atipicos del metodo IQR de la Sesion 4.

In [ ]:
lunes = generador.normal(18000, 5000, 80)
sabado = generador.normal(24000, 3000, 80)
domingo = generador.normal(12000, 7000, 80)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.boxplot([lunes, sabado, domingo])

ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['Lunes', 'Sabado', 'Domingo'])

ax.set_title('Gasto por dia de la semana')
ax.set_ylabel('Gasto (pesos)')
ax.grid(axis='y', alpha=0.3)
plt.show()

---
## Parte 4 - Personalizacion: de un grafico a una figura presentable

| Elemento | Metodo |
|---|---|
| Titulo | `ax.set_title('...')` |
| Nombre del eje x | `ax.set_xlabel('...')` |
| Nombre del eje y | `ax.set_ylabel('...')` |
| Leyenda | `ax.legend()` mas `label=` al dibujar |
| Rango del eje | `ax.set_ylim(0, algo)` |
| Rejilla | `ax.grid(axis='y', alpha=0.3)` |
| Valor sobre cada barra | `ax.bar_label(barras)` |
| Colores | `color=` o `color=[lista]` |

`ax.legend()` solo muestra lo que tenga `label=`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

barras = ax.bar(
    categorias,
    unidades,
    color=['#1A56DB', '#0891B2', '#7C3AED', '#B45309'],
    edgecolor='black',
    linewidth=0.6,
    label='Unidades vendidas 2026'
)

ax.set_title('Unidades vendidas por categoria', fontsize=14, fontweight='bold')
ax.set_xlabel('Categoria')
ax.set_ylabel('Unidades vendidas')
ax.set_ylim(0, max(unidades) * 1.2)
ax.bar_label(barras, padding=3, fontsize=10)
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

### 4.1 Los nombres de los metodos hay que escribirlos exactos

| Se escribe | No existe |
|---|---|
| `ax.set_title()` | `ax.set_titulo()`, `ax.title()` |
| `ax.set_xlabel()` | `ax.set_xlabels()`, `ax.xlabel()` |
| `ax.legend()` | `ax.leyenda()`, `ax.set_legend()` |

En VS Code, escribe `ax.` y presiona **Tab** para ver todo lo que ese objeto sabe hacer.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(categorias, unidades, color='#1A56DB')
ax.set_titulo('Unidades vendidas por categoria')
ax.set_ylabel('Unidades')
plt.show()

### 4.2 `bar_label` necesita las barras, no los datos

`barras = ax.bar(...)` no guarda los numeros: guarda los rectangulos ya dibujados.
`bar_label` recorre esos rectangulos para saber donde poner cada texto.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.bar(categorias, unidades, color='#0891B2')

ax.bar_label(unidades, padding=3)

ax.set_title('Unidades por categoria, con el valor encima')
ax.set_ylim(0, max(unidades) * 1.15)
plt.show()

---
## Parte 5 - Del `groupby` al grafico: los datos del Titanic

El dataset viene con seaborn. Si no hay internet, la celda arma una tabla de respaldo
equivalente.

In [ ]:
def titanic_de_respaldo():
    rng = np.random.default_rng(2026)
    n = 400
    pclass = rng.choice([1, 2, 3], size=n, p=[0.24, 0.21, 0.55])
    sexo = np.where(rng.random(n) < 0.35, 'female', 'male')
    sexo[0] = 'male'
    prob = 0.10 + 0.55 * (sexo == 'female') + 0.15 * (3 - pclass)
    survived = (rng.random(n) < np.clip(prob, 0.03, 0.97)).astype(int)
    age = np.clip(rng.normal(30 - 2 * (pclass - 1), 13, n), 0.5, 80).round(1)
    fare = np.clip(rng.gamma(2.2, 1.0, n) * (90 / pclass ** 2), 3, 300).round(2)
    return pd.DataFrame({
        'survived': survived, 'pclass': pclass, 'sex': sexo,
        'age': age, 'fare': fare,
        'embark_town': rng.choice(['Southampton', 'Cherbourg', 'Queenstown'], n)
    })


def cargar_titanic():
    try:
        return sns.load_dataset('titanic')
    except Exception:
        print('Sin internet: uso la tabla de respaldo construida a mano.')
        return titanic_de_respaldo()


df = cargar_titanic()

print('Filas y columnas:', df.shape)
print(df[['survived', 'pclass', 'sex', 'age', 'fare']].head())

### 5.1 La tabla y el grafico, uno al lado del otro

`survived` vale 1 o 0, asi que el promedio de esa columna es la tasa de supervivencia.

In [ ]:
tasa = df.groupby('pclass')['survived'].mean()

print('La tabla:')
print(tasa.round(3))
print()

fig, ax = plt.subplots(figsize=(7, 4.5))

etiquetas = ['Clase ' + str(c) for c in tasa.index]
barras = ax.bar(etiquetas, tasa.values, color=['#1A56DB', '#0891B2', '#B45309'])
ax.bar_label(barras, labels=[f'{v:.0%}' for v in tasa.values], padding=3, fontsize=11)

ax.set_title('Tasa de supervivencia por clase', fontsize=13, fontweight='bold')
ax.set_ylabel('Proporcion que sobrevivio')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()

### 5.2 Lee el grafico en voz alta

La tabla decia 0.63, 0.47, 0.24. El grafico dice **una escalera que baja**. Pasar del
numero a la explicacion es la interpretacion critica que evalua la fase de visualizacion del Proyecto (Actividad 3).

---
## Parte 6 - Seaborn: lo mismo con menos codigo

1. Las funciones de seaborn reciben `data=`, `x=`, `y=` con **nombres de columna**.
2. Casi todas aceptan `ax=ax`: dibujas seaborn dentro de tu propio marco y lo personalizas
   con los metodos de matplotlib.
3. `hue='columna'` colorea por una tercera variable y arma la leyenda sola.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(data=df, x='age', bins=20, kde=True, color='#1A56DB', ax=ax)

ax.set_title('Distribucion de edades de los pasajeros', fontsize=13, fontweight='bold')
ax.set_xlabel('Edad (anios)')
ax.set_ylabel('Cuantos pasajeros')
ax.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=df, x='pclass', y='age', ax=ax)

ax.set_title('Distribucion de la edad por clase', fontsize=13, fontweight='bold')
ax.set_xlabel('Clase del tiquete')
ax.set_ylabel('Edad (anios)')
plt.show()

### 6.1 Los nombres de columna se escriben exactos

Antes de graficar, imprime `list(df.columns)` y copia el nombre exacto desde ahi.

In [ ]:
print(list(df.columns))

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df, x='columna_inexistente', y='fare', hue='survived', alpha=0.6, ax=ax)

ax.set_title('Edad contra tarifa pagada', fontsize=13, fontweight='bold')
ax.set_xlabel('Edad (anios)')
ax.set_ylabel('Tarifa pagada')
ax.legend(title='Sobrevivio')
plt.show()

### 6.2 El mapa de calor de correlaciones

`df.corr()` calcula la correlacion de todas las columnas contra todas; `sns.heatmap()` la
pinta. La correlacion es aritmetica: con columnas de texto no se puede calcular.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))

matriz = df.corr()

sns.heatmap(matriz, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f', ax=ax)

ax.set_title('Correlacion entre las variables numericas', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Parte 7 - El grafico que cuenta toda la historia

`sns.barplot(data=df, x='pclass', y='survived', hue='sex')` agrupa por clase y por sexo,
promedia, dibuja y arma la leyenda, todo en una linea.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))

sns.barplot(data=df, x='pclass', y='survived', hue='sex',
            errorbar=None, palette=['#1A56DB', '#B45309'], ax=ax)

ax.set_title('Supervivencia por clase y sexo', fontsize=14, fontweight='bold')
ax.set_xlabel('Clase del tiquete')
ax.set_ylabel('Proporcion que sobrevivio')
ax.set_ylim(0, 1)
ax.legend(title='Sexo')
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for contenedor in ax.containers:
    ax.bar_label(contenedor, fmt='%.2f', padding=2, fontsize=9)

plt.tight_layout()
plt.show()

### 7.1 Interpretacion critica: que dice esta figura

Escribelo con tus palabras. La estructura que busca la rubrica:

1. **Que se ve** (el hecho visual, sin interpretar)
2. **Que significa** (la traduccion al fenomeno real)
3. **Que no se puede concluir** (el limite honesto de tus datos)

**Tu interpretacion:**



---
## Parte 8 - Como esto alimenta tu Proyecto (Actividad 3)

La visualizacion es una fase del **Proyecto (Actividad 3, 40%)**. La rubrica valora (criterios de 25%):

| Criterio | Donde lo viste hoy |
|---|---|
| Personalizacion avanzada | Parte 4 |
| Graficos variados | Parte 3 |
| Interactividad con Plotly | **Sesion 6** |
| Interpretacion critica | Partes 1.3, 5.2 y 7.1 |

**El entorno debe tener matplotlib, seaborn Y plotly.** Plotly llega el 29 de septiembre.

- [ ] Toda figura tiene titulo
- [ ] Todo eje tiene nombre y unidades
- [ ] Si hay mas de un color, hay leyenda
- [ ] Cada grafico va seguido de su interpretacion
- [ ] Hay al menos un grafico de plotly interactivo
- [ ] El notebook corre de arriba a abajo sin errores

---
---
## TAREA - antes del martes 29 de septiembre

Es el ensayo directo de la fase de visualizacion del Proyecto (Actividad 3). Usa el dataset `tips` (cuentas y propinas de un
restaurante), que trae seaborn. La celda de abajo lo carga con el mismo paracaidas offline
de la Parte 5.

### Parte 1 - Cinco graficos con su interpretacion

In [ ]:
def tips_de_respaldo():
    rng = np.random.default_rng(11)
    n = 244
    total_bill = np.clip(rng.gamma(5.0, 4.0, n), 3, 60).round(2)
    tip = np.clip(total_bill * rng.normal(0.16, 0.05, n), 1, 12).round(2)
    return pd.DataFrame({
        'total_bill': total_bill, 'tip': tip,
        'sex': rng.choice(['Male', 'Female'], n),
        'day': rng.choice(['Thur', 'Fri', 'Sat', 'Sun'], n),
        'time': rng.choice(['Lunch', 'Dinner'], n),
        'size': rng.choice([1, 2, 3, 4, 5], n)
    })


def cargar_tips():
    try:
        return sns.load_dataset('tips')
    except Exception:
        print('Sin internet: uso la tabla de respaldo.')
        return tips_de_respaldo()


tips = cargar_tips()
print(tips.shape)
print(list(tips.columns))
print(tips.head())

# TODO 1: histograma de 'total_bill' con sns.histplot. Prueba dos valores de bins.
# TODO 2: barras de la propina promedio por dia (sns.barplot, x='day', y='tip').
# TODO 3: dispersion de 'total_bill' contra 'tip', coloreada por 'time'.
# TODO 4: boxplot de 'tip' por dia. Donde estan los atipicos?
# TODO 5: mapa de calor de correlacion de las columnas numericas.
#
# REGLAS DE ENTREGA (son las de la rubrica):
#   - Cada figura se crea con fig, ax = plt.subplots()
#   - Cada figura lleva set_title, set_xlabel y set_ylabel con unidades
#   - Debajo de cada grafico, una celda de texto de 2 o 3 renglones interpretandolo

### Parte 2 - Responde

1. Que es `fig` y que es `ax`? Por que se prefiere `fig, ax = plt.subplots()` a `plt.plot()`?
2. Que pregunta responde un histograma que no responde un grafico de barras?
3. Por que no se debe usar un grafico de linea para comparar categorias?
4. Que ensena el cuarteto de Anscombe en una frase?
5. Que hace `hue=` en seaborn y que trabajo te ahorra?
6. Por que `df.corr()` puede reventar y `df[columnas_numericas].corr()` no?

**Tus respuestas:**

1.
2.
3.
4.
5.
6.

---
## Resumen

| Que quieres | Con matplotlib | Con seaborn |
|---|---|---|
| Crear la figura y el marco | `fig, ax = plt.subplots(figsize=(8,4))` | (se le pasa `ax=ax`) |
| Comparar categorias | `ax.bar(etiquetas, valores)` | `sns.barplot(data=df, x=, y=)` |
| Evolucion en el tiempo | `ax.plot(x, y, marker='o')` | `sns.lineplot(data=df, x=, y=)` |
| Relacion entre dos numericas | `ax.scatter(x, y, alpha=0.6)` | `sns.scatterplot(data=df, x=, y=, hue=)` |
| Distribucion de una numerica | `ax.hist(datos, bins=15)` | `sns.histplot(data=df, x=, kde=True)` |
| Grupos y atipicos | `ax.boxplot([g1, g2, g3])` | `sns.boxplot(data=df, x=, y=)` |
| Correlaciones de un vistazo | (a mano) | `sns.heatmap(matriz, annot=True)` |
| Titulo y etiquetas | `ax.set_title()`, `ax.set_xlabel()`, `ax.set_ylabel()` | los mismos, sobre el `ax` |
| Leyenda | `label=` al dibujar y luego `ax.legend()` | `hue=` la arma sola |
| Valor sobre las barras | `ax.bar_label(barras, padding=3)` | igual, usando `ax.containers` |

### Los cinco errores que viste hoy

| Error | Que significa |
|---|---|
| `ValueError: x and y must have same first dimension` | las dos listas no tienen el mismo largo |
| `AttributeError: 'Axes' object has no attribute ...` | ese metodo no existe: revisa como se escribe |
| `AttributeError: 'list' object has no attribute 'patches'` | le pasaste datos donde iban barras dibujadas |
| `ValueError: Could not interpret value ... for x` | ese nombre de columna no esta en la tabla |
| `ValueError: could not convert string to float` | pediste una cuenta sobre una columna de texto |

### Las tres reglas que no hay que olvidar

1. **Siempre `fig, ax = plt.subplots()`.** Sabes sobre que dibujas.
2. **Ningun grafico sale sin titulo ni etiquetas de eje.** Es el 25% de la rubrica.
3. **Todo grafico va seguido de una frase que lo interpreta.** Es otro 25%.

**Proxima sesion:** Plotly - los mismos graficos, pero interactivos, y el EDA completo.

*Dudas: consulta al profesor*

---
---
## Ejercicios de refuerzo (opcionales)

Practica por tu cuenta con el dataset `tips` que cargaste en la tarea. Van de mas facil a
mas retador. Intenta primero sin mirar la pista.

### Ejercicio 1 - Facil
Haz un histograma de `total_bill` con titulo, nombre en los dos ejes y 20 cajones.

> *Pista: `fig, ax = plt.subplots()` y despues `sns.histplot(data=tips, x='total_bill', bins=20, ax=ax)`*

In [ ]:
# Tu solucion aqui


### Ejercicio 2 - Medio
Grafico de barras de la propina promedio por dia, con el valor exacto encima de cada barra
y el eje y empezando en cero.

> *Pista: `sns.barplot(data=tips, x='day', y='tip', errorbar=None, ax=ax)` y luego recorre
> `ax.containers` con `ax.bar_label()`.*

In [ ]:
# Tu solucion aqui


### Ejercicio 3 - Reta
Arma **una sola figura con dos marcos** (`plt.subplots(1, 2)`): a la izquierda la dispersion
de `total_bill` contra `tip` coloreada por `time`; a la derecha el boxplot de `tip` por dia.
Ponle un titulo general a la figura y un titulo distinto a cada marco.

> *Pista: `fig, ejes = plt.subplots(1, 2, figsize=(13, 5))`, pasale `ax=ejes[0]` y
> `ax=ejes[1]` a las funciones de seaborn, y usa `fig.suptitle()` para el titulo general.*

In [ ]:
# Tu solucion aqui
